<a href="https://colab.research.google.com/github/Kusanagi1984/Unicafam_Modelos_Analiticos_Indra/blob/main/Sesion3_Integracion_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Integración de datos**

In [1]:
import pandas as pd
import numpy as np
from io import StringIO

In [2]:
csv_clientes = """id_cliente;nombre;ciudad;segmento
C001;Ana Ruiz;Bogotá;Oro
C002;Luis Peña;Medellín;Plata
C003;Marta Gil;Cali;Oro
C004;Juan Ríos;Bogotá;Bronce
C005;Sofía Mora;Barranquilla;Plata"""



In [3]:
df_california_housing = pd.read_csv("/content/sample_data/california_housing_train.csv")

In [4]:
df_california_housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0
...,...,...,...,...,...,...,...,...,...
16995,-124.26,40.58,52.0,2217.0,394.0,907.0,369.0,2.3571,111400.0
16996,-124.27,40.69,36.0,2349.0,528.0,1194.0,465.0,2.5179,79000.0
16997,-124.30,41.84,17.0,2677.0,531.0,1244.0,456.0,3.0313,103600.0
16998,-124.30,41.80,19.0,2672.0,552.0,1298.0,478.0,1.9797,85800.0


In [5]:

clientes = pd.read_csv(StringIO(csv_clientes), sep=";")


In [6]:
print(clientes.shape)
print(clientes.head(3))

(5, 4)
  id_cliente     nombre    ciudad segmento
0       C001   Ana Ruiz    Bogotá      Oro
1       C002  Luis Peña  Medellín    Plata
2       C003  Marta Gil      Cali      Oro


In [7]:
respuesta_api = [
    {"pedido": 101, "cliente": "C001", "items": 3,
     "pago": {"medio": "Tarjeta", "valor": 150000}},
    {"pedido": 102, "cliente": "C003", "items": 1,
     "pago": {"medio": "PSE", "valor": 80000}},
    {"pedido": 103, "cliente": "C001", "items": 2,
     "pago": {"medio": "Efectivo", "valor": 45000}},
    {"pedido": 104, "cliente": "C009", "items": 5,
     "pago": {"medio": "Tarjeta", "valor": 210000}},
]
pedidos = pd.json_normalize(respuesta_api)
print(pedidos)


   pedido cliente  items pago.medio  pago.valor
0     101    C001      3    Tarjeta      150000
1     102    C003      1        PSE       80000
2     103    C001      2   Efectivo       45000
3     104    C009      5    Tarjeta      210000


**consolidación de datos**

In [8]:
np.random.seed(1)
def ventas_mes(mes):
    return pd.DataFrame({
        "mes": mes,
        "sucursal": ["Norte", "Sur", "Centro"],
        "ventas": np.random.randint(80, 200, 3) * 1e6})



In [9]:
ene = ventas_mes("Ene")
ene

,mes,sucursal,ventas
0,Ene,Norte,117000000.0
1,Ene,Sur,187000000.0
2,Ene,Centro,92000000.0


In [10]:
feb = ventas_mes("Feb")
feb

,mes,sucursal,ventas
0,Feb,Norte,152000000.0
1,Feb,Sur,89000000.0
2,Feb,Centro,155000000.0


In [11]:
mar = ventas_mes("Mar")
mar

,mes,sucursal,ventas
0,Mar,Norte,85000000.0
1,Mar,Sur,159000000.0
2,Mar,Centro,144000000.0


In [12]:
consolidado = pd.concat([ene,feb,mar], ignore_index=True)
print(consolidado.shape)

print(consolidado.tail(3))


(9, 3)
   mes sucursal       ventas
6  Mar    Norte   85000000.0
7  Mar      Sur  159000000.0
8  Mar   Centro  144000000.0


In [13]:
consolidado

,mes,sucursal,ventas
0,Ene,Norte,117000000.0
1,Ene,Sur,187000000.0
2,Ene,Centro,92000000.0
3,Feb,Norte,152000000.0
4,Feb,Sur,89000000.0
5,Feb,Centro,155000000.0
6,Mar,Norte,85000000.0
7,Mar,Sur,159000000.0
8,Mar,Centro,144000000.0


**Ejercicio 1: Clientes inactivos**

In [14]:
# 1. Agregar 3 clientes nuevos (C006–C008) a la tabla clientes

nuevos_clientes_csv="""id_cliente;nombre;ciudad;segmento
C006;Carlos Gomez;Cali;Bronce
C007;Elena Perez;Medellín;Plata
C008;Pedro Diaz;Bogotá;Oro"""

nuevos_clientes = pd.read_csv(StringIO(nuevos_clientes_csv), sep=";")

clientes = pd.concat([clientes, nuevos_clientes], ignore_index=True)
print("--- Clientes Actualizados ---")
display(clientes)



--- Clientes Actualizados ---


,id_cliente,nombre,ciudad,segmento
0,C001,Ana Ruiz,Bogotá,Oro
1,C002,Luis Peña,Medellín,Plata
2,C003,Marta Gil,Cali,Oro
3,C004,Juan Ríos,Bogotá,Bronce
4,C005,Sofía Mora,Barranquilla,Plata
5,C006,Carlos Gomez,Cali,Bronce
6,C007,Elena Perez,Medellín,Plata
7,C008,Pedro Diaz,Bogotá,Oro


In [15]:
# 2. Haz un LEFT JOIN con pedidos y lista los clientes sin compras
clientes_pedidos_left = pd.merge(clientes, pedidos, left_on="id_cliente", right_on="cliente", how="left")
clientes_sin_compras = clientes_pedidos_left[clientes_pedidos_left["pedido"].isna()]

print("\n--- Clientes sin compras (LEFT JOIN) ---")
display(clientes_sin_compras[["id_cliente", "nombre", "ciudad", "segmento"]])

# 3. Haz un OUTER JOIN con indicator=True y cuenta cada categoría de _merge
outer_merge = pd.merge(clientes, pedidos, left_on="id_cliente", right_on="cliente", how="outer", indicator=True)
print("\n--- Conteo de categorías en _merge (OUTER JOIN) ---")
print(outer_merge["_merge"].value_counts())

# 4. Escribe la lista final para marketing
# Clientes inactivos (nunca han comprado)
lista_marketing = clientes_sin_compras[["id_cliente", "nombre", "ciudad"]].copy()
print("\n--- Lista final para Marketing (Clientes Inactivos) ---")
display(lista_marketing)



--- Clientes sin compras (LEFT JOIN) ---


,id_cliente,nombre,ciudad,segmento
2,C002,Luis Peña,Medellín,Plata
4,C004,Juan Ríos,Bogotá,Bronce
5,C005,Sofía Mora,Barranquilla,Plata
6,C006,Carlos Gomez,Cali,Bronce
7,C007,Elena Perez,Medellín,Plata
8,C008,Pedro Diaz,Bogotá,Oro



--- Conteo de categorías en _merge (OUTER JOIN) ---
_merge
left_only     6
both          3
right_only    1
Name: count, dtype: int64

--- Lista final para Marketing (Clientes Inactivos) ---


,id_cliente,nombre,ciudad
2,C002,Luis Peña,Medellín
4,C004,Juan Ríos,Bogotá
5,C005,Sofía Mora,Barranquilla
6,C006,Carlos Gomez,Cali
7,C007,Elena Perez,Medellín
8,C008,Pedro Diaz,Bogotá


**Ejercicio 2: Consolidado trimestral**

In [16]:
# 1. Crear tabla pivot (formato ancho: sucursales x meses)
ventas_pivot = consolidado.pivot_table(
    values='ventas',
    index='sucursal',
    columns='mes'
)



In [17]:
# 2. Calcular la columna de crecimiento porcentual entre Enero y Marzo
ventas_pivot['crecimiento_%'] = (
    (ventas_pivot['Mar'] - ventas_pivot['Ene']) / ventas_pivot['Ene']
) * 100


In [18]:

# 3. Ordenar de mayor a menor según el crecimiento
reporte_crecimiento = ventas_pivot.sort_values(by='crecimiento_%', ascending=False)

print("--- Reporte de Ventas y Crecimiento Trimestral ---")
display(reporte_crecimiento)


--- Reporte de Ventas y Crecimiento Trimestral ---


mes,Ene,Feb,Mar,crecimiento_%
sucursal,,,,
Centro,92000000.0,155000000.0,144000000.0,56.521739
Sur,187000000.0,89000000.0,159000000.0,-14.973262
Norte,117000000.0,152000000.0,85000000.0,-27.350427


In [19]:

# 4. Indicar la sucursal líder
sucursal_lider = reporte_crecimiento.index[0]
crecimiento_maximo = reporte_crecimiento['crecimiento_%'].iloc[0]

print(f"\nLa sucursal líder en crecimiento es: **{sucursal_lider}** con un incremento del **{crecimiento_maximo:.2f}%**.")


La sucursal líder en crecimiento es: **Centro** con un incremento del **56.52%**.


**Ejercicio 3: SQL vs pandas**

In [20]:
import sqlite3

# --- 1. Resolución con pandas ---
pandas_resultado = pedidos.groupby("pago.medio")["pago.valor"].sum().reset_index()
pandas_resultado = pandas_resultado.sort_values(by="pago.valor", ascending=False)

print("--- Resultado con pandas ---")
display(pandas_resultado)


--- Resultado con pandas ---


,pago.medio,pago.valor
2,Tarjeta,360000
1,PSE,80000
0,Efectivo,45000


In [21]:

# --- 2. Creación de la tabla en SQLite ---
# Creamos una conexión a una base de datos SQLite en memoria
conn = sqlite3.connect(":memory:")

# Guardamos el DataFrame de pedidos en la base de datos
pedidos.to_sql("pagos", conn, index=False, if_exists="replace")



4

In [22]:
# --- 3. Resolución con SQL ---
query = """
SELECT `pago.medio` AS medio_pago, SUM(`pago.valor`) AS total_ventas
FROM pagos
GROUP BY `pago.medio`
ORDER BY total_ventas DESC;
"""

sql_resultado = pd.read_sql_query(query, conn)
print("\n--- Resultado con SQL (SQLite) ---")
display(sql_resultado)




--- Resultado con SQL (SQLite) ---


,medio_pago,total_ventas
0,Tarjeta,360000
1,PSE,80000
2,Efectivo,45000


In [23]:
# --- 4. Comparación ---
# Cerramos la conexión a la base de datos
conn.close()

print("\n¿Ambos métodos dan el mismo resultado?")
# Verificamos si los valores de ventas ordenados coinciden exactamente
coinciden = (pandas_resultado.iloc[0]["pago.medio"] == sql_resultado.iloc[0]["medio_pago"]) and \
            (pandas_resultado.iloc[0]["pago.valor"] == sql_resultado.iloc[0]["total_ventas"])

if coinciden:
    print(f"¡Sí! Ambos métodos identifican a **{sql_resultado.iloc[0]['medio_pago']}** como el medio de pago líder con un total de **{sql_resultado.iloc[0]['total_ventas']:,}**.")
else:
    print("Se detectaron discrepancias en los resultados.")


¿Ambos métodos dan el mismo resultado?
¡Sí! Ambos métodos identifican a **Tarjeta** como el medio de pago líder con un total de **360,000**.
